# Lesson 9: Running a Pretrained Language Model

In the earlier Weird AI lessons, you built many of the core pieces of a language model from scratch. In this lesson, you will use a pretrained model so you can focus on the complete text-generation pipeline.

The pipeline is:

```text
prompt text -> tokenizer -> token IDs -> pretrained model -> generated token IDs -> generated text
```

By the end of this notebook, you should be able to explain how a prompt becomes token IDs, how a pretrained model generates more tokens, and how those generated tokens become readable text again.

## Section 1: Setup

Run this notebook from the project root folder. If imports fail, make sure your virtual environment is active and the project is installed in editable mode:

```bash
python -m pip install -e .
python -m pip install -r requirements.txt
```

This notebook uses Hugging Face `transformers`. If it is not installed yet, install it with:

```bash
python -m pip install transformers
```

In [1]:
from pathlib import Path
import torch

from weird_ai.pretrained import (
    DEFAULT_MODEL_NAME,
    get_device,
    load_tokenizer,
    load_model,
    tokenize_prompt,
    decode_generated_tokens,
    generate_text,
    generate_with_reasoning,
)

from weird_ai.reasoning import build_reasoning_prompt

## Section 2: Check Your Hardware

A GPU is helpful for larger models, but this lesson should work on CPU with a very small demonstration model. The starter code should choose CUDA when available, Apple Silicon MPS when available, and CPU otherwise.

In [2]:
# TODO: Run this cell after completing get_device() in pretrained.py.

device = get_device()
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.13.0+cu130


## Section 3: Load a Tokenizer

A tokenizer converts human-readable text into token IDs. The model does not directly understand strings such as `Write a parody about networking`. It receives integers.

For this notebook, the default model is intentionally tiny so the first download is fast.

In [3]:
# TODO: Run this cell after completing load_tokenizer() in pretrained.py.

tokenizer = load_tokenizer(DEFAULT_MODEL_NAME)
print(f"Loaded tokenizer for: {DEFAULT_MODEL_NAME}")
print(f"Pad token: {tokenizer.pad_token!r}")
print(f"EOS token: {tokenizer.eos_token!r}")

Loaded tokenizer for: sshleifer/tiny-gpt2
Pad token: '<|endoftext|>'
EOS token: '<|endoftext|>'


## Section 4: Tokenization Round Trip

A round trip means:

```text
text -> token IDs -> text
```

This helps verify that the tokenizer can encode and decode text.

In [4]:
prompt = "Write a parody chorus about a student debugging Python at midnight."

# TODO: Encode the prompt using tokenizer.encode().
token_ids = tokenizer.encode(prompt)

# TODO: Decode the token IDs back into text using tokenizer.decode().
decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(prompt)
print("Token IDs:")
print(token_ids)
print("Decoded text:")
print(decoded_text)

Original text:
Write a parody chorus about a student debugging Python at midnight.
Token IDs:
[16594, 257, 29695, 26441, 546, 257, 3710, 28769, 11361, 379, 15896, 13]
Decoded text:
Write a parody chorus about a student debugging Python at midnight.


### Reflection Check

Answer in your assignment document:

Why does the model need token IDs instead of regular text?

## Section 5: Inspect Individual Tokens

Many modern tokenizers use subword tokens. A word may be one token, or it may be split into multiple pieces.

In [5]:
# TODO: After token_ids contains real token IDs, print each token ID and the text it decodes to.

for token_id in token_ids:
    piece = tokenizer.decode([token_id])
    print(f"{token_id:>6} -> {piece!r}")

 16594 -> 'Write'
   257 -> ' a'
 29695 -> ' parody'
 26441 -> ' chorus'
   546 -> ' about'
   257 -> ' a'
  3710 -> ' student'
 28769 -> ' debugging'
 11361 -> ' Python'
   379 -> ' at'
 15896 -> ' midnight'
    13 -> '.'


## Section 6: Load a Pretrained Model

A pretrained model already has learned statistical patterns from a large training corpus. In this lesson, you are not training the model. You are using it for inference.

This can take a moment the first time because the model weights need to download.

In [6]:
# TODO: Run this cell after completing load_model() in pretrained.py.

model = load_model(DEFAULT_MODEL_NAME, device=device)
print(f"Loaded model for: {DEFAULT_MODEL_NAME}")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model for: sshleifer/tiny-gpt2


## Section 7: Generate Text

Now you will complete the full pipeline:

```text
prompt -> tokenizer -> model.generate() -> decode
```

The `generate_text()` function in `pretrained.py` should handle those steps.

In [7]:
# TODO: Run this cell after completing generate_text() in pretrained.py.

prompt = "Write a short parody chorus about forgetting to commit code before a demo:"

output = generate_text(
    prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(output)

Outside 236 perhaps omega Televisionobl rubbing skillet 236 Television Pocket braveryOutside grandchildrenMini TreozygSexual� TreProspublicSexual predatorsMini Boone mutualOutside Boone clearerpublic Wheels omega factors Pocket perhaps Medic soy clearer grandchildren rentedpublic perhaps omegaobl rubbing deflectProsMini equate


## Section 8: Compare Temperature Settings

Temperature changes how random generation is.

Lower temperature usually produces safer, more predictable text. Higher temperature usually produces more varied, surprising, and sometimes less coherent text.

In [8]:
prompt = "Write one funny line about a Wi-Fi router with stage fright:"

temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    print("=" * 70)
    print(f"Temperature: {temp}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=temp,
        top_k=50,
        device=device,
    ))
    print()

Temperature: 0.3
Most brutalityobl brutality Singapore clearer courtyard Singapore 236 omega workshopsivedPros Television Wheels praying rentedOutsidePros SingaporePros rubbingMini deflectSexual grandchildrenMiniGy incarcerSexual equateaciousacious Television Tre incarcer lined Pocket Tre predators

Temperature: 0.8
 236ived praying courtyard rubbing Wheels workshops Television equate Dreams Dreamspublic praying Boone rented Tre Redux Treshows grandchildren Television653 lined courtyard praying Singapore mutual perhaps Wheels Singapore Television clearer factors Dreams representations linedGy omega equate representations

Temperature: 1.2
 236 Television brutalityshows brutality lined Late Boone Pocket rented skillet 236 incarcer skillet equate rubbing perhaps skillet rubbing grandchildren ReduxGySexual448 membership praying448 predators Dreams membership Boone ReduxPros deflectMost workshops membership brutalityMost skillet



## Section 9: Compare Top-k Settings

Top-k sampling limits the model to choosing from only the `k` most likely next tokens.

Smaller values restrict the model more. Larger values allow more variety.

In [11]:
prompt = "Write one strange parody lyric about a database that sings backup vocals:"

top_k_values = [10, 50, 100]

for k in top_k_values:
    print("=" * 70)
    print(f"Top-k: {k}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=0.8,
        top_k=k,
        device=device,
    ))
    print()

Top-k: 10
 mutual clearer lined mutual lined Boone Wheels lined bravery lined Boone factors deflect lined deflect Television boils boils deflect factors clearer Television clearer mutual boils factors bravery Television Television factors Television Boone mutual factors deflect clearer mutual Wheels boils boils

Top-k: 50
showsMost predators rubbing BooneMost linedPros boils rubbing 236 membershipoblOutside mutual Television Pocket Singapore brutalityPros deflect Tre membershipshows Wheels incarcer 236Mini workshops brutality�� TreMini Medic deflect deflectMost Late Late

Top-k: 100
 Bendワ Medic Dante quot eruption clearer Booneobl Gandhibalanced imper caveats representationsSexual Flowers brutality MedicechoMQ caveats workshopsobl believesatable caveats brutality workshopsinen Mana equate bravery caveats lined membership Usage brutality Pocket skillet Damn



## Section 10: Generate with a Reasoning Prompt

In Lesson 8, you built prompt helpers that ask Weird AI to think step-by-step. Now you can combine that reasoning prompt with a pretrained model.

In [12]:
user_prompt = "Create a parody song idea about students learning recursion."

reasoning_prompt = build_reasoning_prompt(user_prompt)
print(reasoning_prompt)

You are Weird AI.
Your task is to think through the user's prompt. Think step-by-step:
Create a parody song idea about students learning recursion.
After clearly understanding the user's prompt, produce the Final Answer.


In [13]:
# TODO: Run this cell after completing generate_with_reasoning() in pretrained.py.

reasoning_output = generate_with_reasoning(
    user_prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(reasoning_output)

 perhaps448 Singapore Bend courtyard representations workshops brutality Dreams448public Singapore workshops Tre skillet MedicSexual equate rented boils Bend lined equate Bend mutual equate PocketProsMost lined factors Late courtyardaciouspublic deflect SingaporeMiniSexual perhaps braverypublic skillet Dreams Redux448 clearerSexual Dreams Pocket� Late Television653Gy Television Television membership Wheelsacious 236 incarcer lined omega deflectobl grandchildren MedicGy Boone Medicobl TreOutside omega brutality incarcer membershipPros448 brutality factors Wheels Pocketoblshows Lateozygozyg incarcer WheelsGy grandchildren clearer boils Medic Dreams courtyardPros�


## Section 11: Weird AI Experiment

Choose one of your own parody topics. Generate at least two outputs using different settings. Save the prompts, settings, and outputs in your assignment document.

In [16]:
# TODO: Replace this with your own topic.
my_topic = "replacing AI agents with people (how does it feel to get replaced)"

# TODO: Try at least two combinations of temperature and top_k.
settings = [
    {"temperature": 0.25, "top_k": 15},
    {"temperature": 1.2, "top_k": 100},
]

for setting in settings:
    print("=" * 70)
    print(setting)
    print("=" * 70)
    print(generate_text(
        f"Write a short parody verse about {my_topic}:",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=80,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        device=device,
    ))
    print()

{'temperature': 0.25, 'top_k': 15}
 Wheels Wheels boilsozyg 236 boils 236 236 predators predators predators Wheels deflect clearer 236 Wheels bravery mutual Television Bend factors bravery Boone Wheels deflect Boone clearer deflect lined Boone braveryozyg Bend mutual boils Wheels bravery deflect boils boils Bend factors lined mutual Boone Booneozyg Boone deflect factors mutual mutual clearer Boone Bend boils mutualSexual factors predators mutual 236 236 factorsSexual mutual Wheels Bend TelevisionozygSexual bravery bravery factors WheelsSexual linedSexual 236 clearer

{'temperature': 1.2, 'top_k': 100}
 disguinenOutside bravery thanks lined )) rubbing boils Dreams letters mutual�Secure proxyMQ Television bravery Wheels praying thanks Wheels factorsMQ Usage fathers representations deflectived Dreams caveatsobl Feepublic cube Onlyivedlicts Singaporeived imper brutality Gandhi deflect Manaozyg rubbing mutual hugs clearer deflect c proxy Pocketbalanced UsageImpro Undead boils�545 soySecure 

## Section 12: Lesson Summary

In this notebook, you practiced:

- Loading a pretrained tokenizer
- Encoding text into token IDs
- Decoding token IDs back into text
- Loading a pretrained causal language model
- Generating text from a prompt
- Comparing temperature and top-k settings
- Reusing a Lesson 8 reasoning prompt with a pretrained model

Before submitting, run the unit tests:

```bash
python -m pytest tests/test_pretrained.py
```